In [104]:
import torch
import copy
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

### Cargar California housing dataset y dividirlo

In [105]:
torch.manual_seed(42)

data = fetch_california_housing()

X = torch.tensor(data.data, dtype=torch.float32)
y = torch.tensor(data.target, dtype=torch.float32).view(-1, 1)

feature_names = data.feature_names
print(feature_names)
print(X.shape, y.shape)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
torch.Size([20640, 8]) torch.Size([20640, 1])
Train: torch.Size([16512, 8]) torch.Size([16512, 1])
Val: torch.Size([2064, 8]) torch.Size([2064, 1])
Test: torch.Size([2064, 8]) torch.Size([2064, 1])


### Prepocesamiento

In [106]:
class Preprocessor(nn.Module):
    def __init__(self, feature_names):
        super().__init__()
        self.feature_names = feature_names
        
        self.lat_idx = feature_names.index("Latitude")
        self.lon_idx = feature_names.index("Longitude")
        self.keep_idx = [self.lat_idx, self.lon_idx]
        self.scale_idx = [i for i in range(len(feature_names)) if i not in self.keep_idx]
        
        self.register_buffer("lower", torch.zeros(len(self.scale_idx)))
        self.register_buffer("upper", torch.zeros(len(self.scale_idx)))
        self.register_buffer("mean", torch.zeros(len(self.scale_idx)))
        self.register_buffer("std", torch.ones(len(self.scale_idx)))

    def fit(self, X):
        X_scale = X[:, self.scale_idx]
        
        q1 = torch.quantile(X_scale, 0.25, dim=0)
        q3 = torch.quantile(X_scale, 0.75, dim=0)
        iqr = q3 - q1
        
        self.lower = q1 - 1.5 * iqr
        self.upper = q3 + 1.5 * iqr
        
        mask = ((X_scale >= self.lower) & (X_scale <= self.upper)).all(dim=1)
        
        X_clean = X[mask]
        X_clean_scale = X_clean[:, self.scale_idx]
        
        self.mean = X_clean_scale.mean(dim=0)
        self.std = X_clean_scale.std(dim=0, unbiased=False)
        self.std[self.std == 0] = 1.0
        
        return mask

    def transform(self, X):
        X_new = X.clone()
        X_new[:, self.scale_idx] = (X_new[:, self.scale_idx] - self.mean) / self.std
        return X_new

    def forward(self, X):
        return self.transform(X)

### Eliminar outliers en train

In [107]:
preprocessor = Preprocessor(feature_names)

train_mask = preprocessor.fit(X_train)

X_train = X_train[train_mask]
y_train = y_train[train_mask]

X_train = preprocessor(X_train)
X_val = preprocessor(X_val)
X_test = preprocessor(X_test)

print("Train after outlier removal:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train after outlier removal: torch.Size([13460, 8]) torch.Size([13460, 1])
Val: torch.Size([2064, 8]) torch.Size([2064, 1])
Test: torch.Size([2064, 8]) torch.Size([2064, 1])


In [108]:
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=64, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=64, shuffle=False)

### Red

In [109]:
class RegressionNet(nn.Module):
    def __init__(self, input_size, hidden_sizes, dropout=0.0):
        super().__init__()
        
        layers = []
        prev_size = input_size
        
        for h in hidden_sizes:
            layers.append(nn.Linear(prev_size, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_size = h
        
        layers.append(nn.Linear(prev_size, 1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

### Funcion de training

In [110]:
from tqdm import tqdm

def train_model(model, train_loader, val_loader, lr, epochs):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    best_val_loss = float("inf")
    best_state = None
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        for X_batch, y_batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            optimizer.zero_grad()
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * X_batch.size(0)
        
        train_loss = train_loss / len(train_loader.dataset)
        
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                preds = model(X_batch)
                loss = criterion(preds, y_batch)
                val_loss += loss.item() * X_batch.size(0)
        
        val_loss = val_loss / len(val_loader.dataset)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
        
        print(f"Epoch {epoch+1}/{epochs} - train: {train_loss:.4f} | val: {val_loss:.4f}")
    
    model.load_state_dict(best_state)
    return model, best_val_loss

### Funcion de evaluacion

In [111]:
def evaluate_model(model, loader):
    model.eval()
    criterion = nn.MSELoss()
    
    total_loss = 0.0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            
            all_preds.append(preds)
            all_targets.append(y_batch)
    
    mse = total_loss / len(loader.dataset)
    preds = torch.cat(all_preds, dim=0)
    targets = torch.cat(all_targets, dim=0)
    mae = torch.mean(torch.abs(preds - targets)).item()
    rmse = torch.sqrt(torch.tensor(mse)).item()
    
    return mse, rmse, mae

### 3 Modelos


In [112]:
configs = [
    {"name": "Model 1", "hidden_sizes": [32, 16], "dropout": 0.0, "lr": 0.001, "epochs": 20},
    {"name": "Model 2", "hidden_sizes": [64, 32], "dropout": 0.1, "lr": 0.001, "epochs": 20},
    {"name": "Model 3", "hidden_sizes": [128, 64], "dropout": 0.2, "lr": 0.0005, "epochs": 20}
]

best_model = None
best_name = None
best_val_loss = float("inf")
results = []

input_size = X_train.shape[1]

for config in configs:
    print(f"\nTraining {config['name']}")
    
    model = RegressionNet(
        input_size=input_size,
        hidden_sizes=config["hidden_sizes"],
        dropout=config["dropout"]
    )
    
    model, val_loss = train_model(
        model,
        train_loader,
        val_loader,
        lr=config["lr"],
        epochs=config["epochs"]
    )
    
    mse, rmse, mae = evaluate_model(model, val_loader)
    
    results.append({
        "name": config["name"],
        "val_mse": mse,
        "val_rmse": rmse,
        "val_mae": mae
    })
    
    print(f"{config['name']} - Val MSE: {mse:.4f}, Val RMSE: {rmse:.4f}, Val MAE: {mae:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model = model
        best_name = config["name"]


Training Model 1


Epoch 1/20: 100%|██████████| 211/211 [00:00<00:00, 650.05it/s]


Epoch 1/20 - train: 1.6851 | val: 1.0026


Epoch 2/20: 100%|██████████| 211/211 [00:00<00:00, 700.94it/s]


Epoch 2/20 - train: 0.7907 | val: 0.7956


...

Epoch 18/20 - train: 0.5441 | val: 0.7597


Epoch 19/20: 100%|██████████| 211/211 [00:00<00:00, 514.12it/s]


Epoch 19/20 - train: 0.5328 | val: 0.7984


Epoch 20/20: 100%|██████████| 211/211 [00:00<00:00, 532.83it/s]

Epoch 20/20 - train: 0.5209 | val: 0.7781
Model 3 - Val MSE: 0.5729, Val RMSE: 0.7569, Val MAE: 0.5652


### Resultados

In [113]:
for r in results:
    print(r)
    
print("\nBest model:", best_name)

test_mse, test_rmse, test_mae = evaluate_model(best_model, test_loader)

print("Best model (test):", best_name)
print(f"Test MSE: {test_mse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test MAE: {test_mae:.4f}")

{'name': 'Model 1', 'val_mse': 0.6413347037263619, 'val_rmse': 0.8008337616920471, 'val_mae': 0.5531468391418457}
{'name': 'Model 2', 'val_mse': 0.7135822255482045, 'val_rmse': 0.8447379469871521, 'val_mae': 0.6064322590827942}
{'name': 'Model 3', 'val_mse': 0.572854151097379, 'val_rmse': 0.7568712830543518, 'val_mae': 0.5652242302894592}

Best model: Model 3
Best model (test): Model 3
Test MSE: 0.5772
Test RMSE: 0.7597
Test MAE: 0.5600
